<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3.3: 高阶函数
**上一节: [插曲: Chisel 标准库](3.2_interlude.ipynb)**<br>
**下一节: [函数式编程](3.4_functional_programming.ipynb)**

## 动机
前一模块中那些烦人的 `for` 循环冗长且违背了函数式编程的目的！在本模块中，你的生成器将变得有趣起来。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 两个 FIR 的故事 <a name="compact-fir"></a>
从上一个模块中，我们的 FIR 滤波器的卷积部分是这样写的：

```scala
val muls = 导线(Vec(length, UInt(8.W)))
for(i <- 0 until length) {
  if(i == 0) muls(i) := io.in * io.consts(i)
  else       muls(i) := regs(i - 1) * io.consts(i)
}

val scan = 导线(Vec(length, UInt(8.W)))
for(i <- 0 until length) {
  if(i == 0) scan(i) := muls(i)
  else scan(i) := muls(i) + scan(i - 1)
}

io.out := scan(length - 1)
```

回顾一下，其思想是将 `io.in` 的每个元素与 `io.consts` 的相应元素相乘，并将结果存储在 `muls` 中。
然后，`muls` 中的元素被累加到 `scan` 中，其中 `scan(0) = muls(0)`，`scan(1) = scan(0) + muls(1) = muls(0) + muls(1)`，一般地 `scan(n) = scan(n-1) + muls(n) = muls(0) + ... + muls(n-1) + muls(n)`。
`scan` 中的最后一个元素（等于所有 `muls` 的总和）被赋给 `io.out`。

然而，对于可能被认为是相当简单的操作来说，这非常冗长。事实上，所有这些都可以写在一行中：

```scala
io.out := (taps zip io.consts).map { case (a, b) => a * b }.reduce(_ + _)
```

它在做什么？！让我们分解一下：
- 假设 `taps` 是所有样本的列表，其中 `taps(0) = io.in`，`taps(1) = regs(0)`，等等。
- `(taps zip io.consts)` 接受两个列表 `taps` 和 `io.consts`，并将它们组合成一个列表，其中每个元素是相应位置输入元素的元组。具体来说，它的值将是 `[(taps(0), io.consts(0)), (taps(1), io.consts(1)), ..., (taps(n), io.consts(n))]`。请记住点号是可选的，因此这等同于 `(taps.zip(io.consts))`。
- `.map { case (a, b) => a * b }` 将匿名函数（接受一个包含两个元素的元组并返回它们的乘积）应用于列表的元素，并返回结果。在这种情况下，结果等同于冗长示例中的 `muls`，其值为 `[taps(0) * io.consts(0), taps(1) * io.consts(1), ..., taps(n) * io.consts(n)]`。你将在下一个模块中重新讨论匿名函数。现在，只需学习这个语法。
- 最后，`.reduce(_ + _)` 也将函数（元素加法）应用于列表的元素。然而，它接受两个参数：第一个是当前累加值，第二个是列表元素（在第一次迭代中，它只是将前两个元素相加）。这些由括号中的两个下划线表示。然后，假设从左到右遍历，结果将是 `(((muls(0) + muls(1)) + muls(2)) + ...) + muls(n)`，首先计算更深嵌套的括号。这就是卷积的输出。

---
# 函数作为参数
正式地说，像 `map` 和 `reduce` 这样的函数被称为_高阶函数_：它们是接受函数作为参数的函数。
事实证明（希望你能从上面的示例中看出），这些是非常强大的构造，封装了通用的计算模式，让你能够专注于应用逻辑而不是控制流，从而产生非常简洁的代码。

## 指定函数的不同方式
You may have noticed that there were two ways of specifying functions in the 示例 above:
- 对于每个参数只被引用一次的函数，你*可能*能够使用下划线（`_`）来引用每个参数。在上面的示例中，`reduce` 参数函数接受两个参数，可以指定为 `_ + _`。虽然方便，但这受到一组额外复杂规则的限制，所以如果它不起作用，请尝试：
- 显式指定输入参数列表。reduce 可以显式地写为 `(a, b) => a + b`，其一般形式是将参数列表放在括号中，后跟 `=>`，然后是引用这些参数的函数体。
- 当需要元组解包时，使用 `case` 语句，如 `case (a, b) => a * b`。这接受单个参数（一个包含两个元素的元组），并将其解包到变量 `a` 和 `b` 中，然后可以在函数体中使用这些变量。

## Scala 实践
在上一模块中，我们已经了解了 Scala 集合 API 中的主要类，例如 `List`s。
这些高阶函数是这些 API 的一部分 - 事实上，上面的示例在 `List`s 上使用了 `map` 和 `reduce` API。
在本节中，我们将通过示例和练习来熟悉这些方法。
在这些示例中，为了简单和清晰起见，我们将在 Scala 数字（`Int`s）上操作，但由于 Chisel 运算符的行为类似，这些概念应该可以推广。

<span style="color:blue">**示例: Map**</span><br>
`List[A].map` 具有类型签名 `map[B](f: (A) ⇒ B): List[B]`。你将在后面的模块中了解更多关于类型的知识。现在，将类型 A 和 B 视为 `Int`s 或 `UInt`s，这意味着它们可以是软件或硬件类型。

简单来说，它接受一个类型为 `(f: (A) ⇒ B)` 的参数，或者一个接受一个类型为 `A` 的参数（与输入列表的元素类型相同）并返回一个类型为 `B` 的值（可以是任何类型）的函数。然后 `map` 返回一个类型为 `B` 的新列表（参数函数的返回类型）。

As we've already explained the behavior of List in the FIR 示例, let's get straight into the 示例 and exercises:

In [ ]:
println(List(1, 2, 3, 4).map(x => x + 1))  // explicit argument list in function
println(List(1, 2, 3, 4).map(_ + 1))  // equivalent to the above, but implicit arguments
println(List(1, 2, 3, 4).map(_.toString + "a"))  // the output element type can be different from the input element type

println(List((1, 5), (2, 6), (3, 7), (4, 8)).map { case (x, y) => x*y })  // this unpacks a tuple, note use of curly braces

// 相关：Scala 有一种构造顺序数字列表的语法
println(0 to 10)  // to 是包含的，端点是结果的一部分
println(0 until 10)  // until 在末端不包含，端点不是结果的一部分

// 这些很大程度上像列表一样行为，并且可以用于生成索引：
val myList = List("a", "b", "c", "d")
println((0 until 4).map(myList(_)))

<span style="color:red">**练习: Map**</span><br><a name="map-练习"></a>

In [ ]:
// 现在你可以尝试：
// 填充空白处（???），使这使输入列表的元素翻倍。
// 这应该返回：List(2, 4, 6, 8)
println(List(1, 2, 3, 4).map(???))

<span style="color:blue">**示例: zipWithIndex**</span><br>
`List.zipWithIndex` 具有类型签名 `zipWithIndex: List[(A, Int)]`。

它不接受参数，但返回一个列表，其中每个元素是原始元素和索引的元组（第一个为零）。
所以 `List("a", "b", "c", "d").zipWithIndex` 将返回 `List(("a", 0), ("b", 1), ("c", 2), ("d", 3))`

当在某个操作中需要元素索引时，这很有用。

因为这非常直观，我们只看一些示例：

In [ ]:
println(List(1, 2, 3, 4).zipWithIndex)  // 注意索引从零开始
println(List("a", "b", "c", "d").zipWithIndex)
println(List(("a", "b"), ("c", "d"), ("e", "f"), ("g", "h")).zipWithIndex)  // 元组嵌套

<span style="color:blue">**示例: Reduce**</span><br>
`List[A].map` 具有与 `reduce(op: (A, A) ⇒ A): A` 类似的类型签名。（实际上它更宽松，`A` 只需要是 List 类型的超类型，但我们这里不处理这种语法）

正如上面已经解释过的，这里有一些示例：

In [ ]:
println(List(1, 2, 3, 4).reduce((a, b) => a + b))  // 返回所有元素的总和
println(List(1, 2, 3, 4).reduce(_ * _))  // 返回所有元素的乘积
println(List(1, 2, 3, 4).map(_ + 1).reduce(_ + _))  // 你可以将 reduce 链接到 map 的结果上

In [ ]:
// 重要提示：reduce 在空列表时将失败
println(List[Int]().reduce(_ * _))

<span style="color:red">**练习: Reduce**</span><br><a name="reduce-练习"></a>

In [ ]:
// 现在你可以尝试：
// 填充空白处（???），使这返回输入列表元素翻倍后的乘积。
// 这应该返回：(1*2)*(2*2)*(3*2)*(4*2) = 384
println(List(1, 2, 3, 4).map(???).reduce(???))

<span style="color:blue">**示例: Fold**</span><br>
`List[A].fold` 与 reduce 非常相似，只是你可以指定初始累加值。
它具有与 `fold(z: A)(op: (A, A) ⇒ A): A` 类似的类型签名。（像 `reduce` 一样，`A` 的类型也更宽松）

值得注意的是，它接受两个参数列表，第一个（`z`）是初始值，第二个是累加函数。
与 `reduce` 不同，它在空列表时不会失败，而是直接返回初始值。

这里有一些示例：

In [ ]:
println(List(1, 2, 3, 4).fold(0)(_ + _))  // 等同于使用 reduce 的总和
println(List(1, 2, 3, 4).fold(1)(_ + _))  // 与上面类似，但累加从 1 开始
println(List().fold(1)(_ + _))  // 与 reduce 不同，不会在空输入时失败

<span style="color:red">**练习: Fold**</span><br><a name="fold-练习"></a>

In [ ]:
// 现在你可以尝试：
// 填充空白处（???），使这返回输入列表元素的双倍乘积。
// 这应该返回：2*(1*2*3*4) = 48
// 注意：除非需要空列表容忍度，否则 reduce 在这里更适合。
println(List(1, 2, 3, 4).fold(???)(???))

<span style="color:red">**练习: Decoupled Arbiter**</span><br>
现在让我们把所有内容整合到一个练习中。

在这个示例中，我们将构建一个 Decoupled 仲裁器：一个具有 _n_ 个 Decoupled 输入和一个 Decoupled 输出的模块。
仲裁器选择最低有效通道并将其转发到输出。

Some hints:
- Architecturally:
  - `io.out.valid` 如果任何输入有效则为 true
  - 考虑有一个内部导线来表示选中的通道
  - 每个输入的 `ready` 在输出准备就绪且该通道被选中时为 true（这在组合逻辑上耦合了 ready 和 valid，但我们现在会忽略这一点...）
- These constructs may help:
  - `map`，特别是用于返回子元素的 Vec，例如 `io.in.map(_.valid)` 返回输入 Bundles 的有效信号列表
  - `PriorityMux(List[Bool, Bits])`，它接受一个有效信号和位的列表，返回第一个有效的元素
  - Vec 上的动态索引，通过 UInt 进行索引，例如 `io.in(0.U)`

In [ ]:
class MyRoutingArbiter(numChannels: Int) extends Module {
  val io = IO(new Bundle {
    val in = Vec(numChannels, Flipped(Decoupled(UInt(8.W))))
    val out = Decoupled(UInt(8.W))
  } )

  // YOUR CODE BELOW
  ???
}

test(new MyRoutingArbiter(4)) { c =>
    // verify that the computation is correct
    // Set input defaults
    for(i <- 0 until 4) {
        c.io.in(i).valid.poke(false.B)
        c.io.in(i).bits.poke(i.U)
        c.io.out.ready.poke(true.B)
    }

    c.io.out.valid.expect(false.B)

    // Check single input valid behavior with backpressure
    for (i <- 0 until 4) {
        c.io.in(i).valid.poke(true.B)
        c.io.out.valid.expect(true.B)
        c.io.out.bits.expect(i.U)

        c.io.out.ready.poke(false.B)
        c.io.in(i).ready.expect(false.B)

        c.io.out.ready.poke(true.B)
        c.io.in(i).valid.poke(false.B)
    }

    // Basic check of multiple input ready behavior with backpressure
    c.io.in(1).valid.poke(true.B)
    c.io.in(2).valid.poke(true.B)
    c.io.out.bits.expect(1.U)
    c.io.in(1).ready.expect(true.B)
    c.io.in(0).ready.expect(false.B)

    c.io.out.ready.poke(false.B)
    c.io.in(1).ready.expect(false.B)
}

println("SUCCESS!!") // Scala Code: if we get here, our tests passed!

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
类 MyRoutingArbiter(numChannels: Int) extends 模块 {
  val io = IO(new 束 {
    val in = Vec(numChannels, Flipped(Decoupled(UInt(8.W))))
    val out = Decoupled(UInt(8.W))
  } )

  // YOUR CODE BELOW
  io.out.valid := io.in.map(\_.valid).reduce(\_ || \_)
  val channel = PriorityMux(
    io.in.map(\_.valid).zipWithIndex.map { case (valid, index) => (valid, index.U) }
  )
  io.out.bits := io.in(channel).bits
  io.in.map(\_.ready).zipWithIndex.foreach { case (ready, index) =>
    ready := io.out.ready && channel === index.U
  }
}
</pre></article></div></section></div>

---
# You're done!

[Return to the top.](#top)